## Importing Libraries

In [3]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('Transaction').getOrCreate()

## Data Cleaning

In [4]:
from pyspark.sql.functions import col, regexp_replace
from pyspark.sql.types import DoubleType

df = spark.read.csv("txn.csv", header=True, inferSchema=False)

df_cleaned = df.withColumn("WITHDRAWAL_AMT", regexp_replace(col(" WITHDRAWAL AMT "), ",", "").cast(DoubleType())) \
               .withColumn("DEPOSIT_AMT", regexp_replace(col(" DEPOSIT AMT "), ",", "").cast(DoubleType())) \
               .withColumn("BALANCE_AMT", regexp_replace(col("BALANCE AMT"), ",", "").cast(DoubleType()))


### Maximum withdrawal amount in transactions

In [6]:
df_cleaned.agg({"WITHDRAWAL_AMT": "max"}).show()

+-------------------+
|max(WITHDRAWAL_AMT)|
+-------------------+
|      4.594475464E8|
+-------------------+



### Minimum withdrawal amount of an account

In [7]:
df_cleaned.groupBy("Account No").agg({"WITHDRAWAL_AMT": "min"}).show()

+-------------+-------------------+
|   Account No|min(WITHDRAWAL_AMT)|
+-------------+-------------------+
|409000438611'|                0.2|
|     1196711'|               0.25|
|     1196428'|               0.25|
|409000493210'|               0.01|
|409000611074'|              120.0|
|409000425051'|               1.25|
|409000405747'|               21.0|
|409000493201'|                2.1|
|409000438620'|               0.34|
|409000362497'|               0.97|
+-------------+-------------------+



### Maximum deposit amount of an account

In [8]:
df_cleaned.groupBy("Account No").agg({"DEPOSIT_AMT": "max"}).show()

+-------------+----------------+
|   Account No|max(DEPOSIT_AMT)|
+-------------+----------------+
|409000438611'|        1.7025E8|
|     1196711'|           5.0E8|
|     1196428'|   2.119594422E8|
|409000493210'|           1.5E7|
|409000611074'|       3000000.0|
|409000425051'|           1.5E7|
|409000405747'|         2.021E8|
|409000493201'|       1000000.0|
|409000438620'|         5.448E8|
|409000362497'|           2.0E8|
+-------------+----------------+



### Minimum deposit amount of an account

In [9]:
df_cleaned.groupBy("Account No").agg({"DEPOSIT_AMT": "min"}).show()

+-------------+----------------+
|   Account No|min(DEPOSIT_AMT)|
+-------------+----------------+
|409000438611'|            0.03|
|     1196711'|            1.01|
|     1196428'|             1.0|
|409000493210'|            0.01|
|409000611074'|          1320.0|
|409000425051'|             1.0|
|409000405747'|           500.0|
|409000493201'|             0.9|
|409000438620'|            0.07|
|409000362497'|            0.03|
+-------------+----------------+



### Sum of balance in every bank account

In [10]:
df_cleaned.groupBy("Account No").agg({"BALANCE_AMT": "sum"}).show()

+-------------+--------------------+
|   Account No|    sum(BALANCE_AMT)|
+-------------+--------------------+
|409000438611'|-2.49486577068339...|
|     1196711'|-1.60476498101275E13|
|     1196428'| -8.1418498130721E13|
|409000493210'|-3.27584952132095...|
|409000611074'|       1.615533622E9|
|409000425051'|-3.77211841164998...|
|409000405747'|-2.43108047067000...|
|409000493201'|1.0420831829499985E9|
|409000438620'|-7.12291867951358...|
|409000362497'| -5.2860004792808E13|
+-------------+--------------------+



### Number of transaction on each date

In [11]:
df_cleaned.groupBy("VALUE DATE").count().orderBy("VALUE DATE").show()

+----------+-----+
|VALUE DATE|count|
+----------+-----+
|  1-Apr-17|    1|
|  1-Aug-15|   75|
|  1-Aug-16|   85|
|  1-Aug-17|   65|
|  1-Aug-18|  144|
|  1-Dec-15|   96|
|  1-Dec-16|  106|
|  1-Dec-17|   45|
|  1-Dec-18|   97|
|  1-Feb-16|   97|
|  1-Feb-17|   81|
|  1-Feb-18|   87|
|  1-Feb-19|   79|
|  1-Jan-15|    3|
|  1-Jan-16|   59|
|  1-Jan-18|   53|
|  1-Jan-19|   57|
|  1-Jul-15|   25|
|  1-Jul-16|  111|
|  1-Jul-17|  243|
+----------+-----+
only showing top 20 rows


### List of customers with withdrawal amount more than 1 lakh

In [12]:
df_cleaned.filter(col("WITHDRAWAL_AMT") > 100000).select("Account No", "WITHDRAWAL_AMT").distinct().show()

+-------------+--------------+
|   Account No|WITHDRAWAL_AMT|
+-------------+--------------+
|409000611074'|      274600.0|
|409000493201'|     1500000.0|
|409000493201'|     199604.27|
|409000438620'|      186604.0|
|409000438620'|   3.6675558E7|
|     1196711'|     7530283.0|
|409000611074'|      145450.0|
|409000493201'|     119401.28|
|     1196711'|      628945.0|
|     1196428'|     2170000.0|
|409000493201'|     239083.95|
|     1196428'|        1.15E7|
|     1196711'|     308204.42|
|409000493201'|      472000.0|
|409000493201'|     199217.29|
|409000438611'|         2.4E8|
|409000493210'|         1.4E7|
|409000493201'|     175473.28|
|409000493201'|     205372.21|
|409000493201'|     131055.59|
+-------------+--------------+
only showing top 20 rows
